# Movies ELK — Exploration & Contrôle qualité

Ce notebook permet de :
- Vérifier l'ingestion dans `movies_raw` et `movies_clean`
- Explorer les données
- Mesurer la qualité avant/après nettoyage

In [ ]:
import os
from elasticsearch import Elasticsearch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connexion à Elasticsearch
ES_URL = os.getenv('ES_URL', 'http://elasticsearch:9200')
es = Elasticsearch(ES_URL)

print('Connexion ES:', es.ping())
print('Infos cluster:', es.info()['version']['number'])

In [ ]:
# ── Vérification du nombre de documents ──────────────────
raw_count   = es.count(index='movies_raw')['count']
clean_count = es.count(index='movies_clean')['count']

print(f'movies_raw   : {raw_count:,} documents')
print(f'movies_clean : {clean_count:,} documents')

In [ ]:
# ── Échantillon movies_clean ──────────────────────────────
resp = es.search(index='movies_clean', size=5, query={'match_all': {}})
for hit in resp['hits']['hits']:
    src = hit['_source']
    print(f"{src.get('title')} ({src.get('release_year')}) — {src.get('genres')} — note: {src.get('vote_average')}")

In [ ]:
# ── Distribution des genres ───────────────────────────────
resp = es.search(index='movies_clean', size=0, aggs={
    'genres': {'terms': {'field': 'genres', 'size': 15}}
})

buckets = resp['aggregations']['genres']['buckets']
df_genres = pd.DataFrame(buckets).rename(columns={'key': 'genre', 'doc_count': 'count'})

plt.figure(figsize=(10, 5))
sns.barplot(data=df_genres, x='count', y='genre', palette='viridis')
plt.title('Distribution des films par genre')
plt.xlabel('Nombre de films')
plt.tight_layout()
plt.show()

In [ ]:
# ── Évolution de la production par année ─────────────────
resp = es.search(index='movies_clean', size=0, aggs={
    'par_annee': {
        'date_histogram': {
            'field': 'release_date',
            'calendar_interval': 'year',
            'format': 'yyyy'
        },
        'aggs': {'avg_vote': {'avg': {'field': 'vote_average'}}}
    }
})

buckets = resp['aggregations']['par_annee']['buckets']
df_annee = pd.DataFrame([{
    'annee': b['key_as_string'],
    'nb_films': b['doc_count'],
    'note_moy': round(b['avg_vote']['value'] or 0, 2)
} for b in buckets if int(b['key_as_string']) >= 1970])

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(df_annee['annee'], df_annee['nb_films'], color='steelblue', alpha=0.6, label='Nb films')
ax2 = ax1.twinx()
ax2.plot(df_annee['annee'], df_annee['note_moy'], color='orange', linewidth=2, label='Note moyenne')
ax1.set_xlabel('Année')
ax1.set_ylabel('Nombre de films')
ax2.set_ylabel('Note moyenne /10')
plt.title('Production cinématographique par année')
plt.xticks(df_annee['annee'][::5], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Mesure qualité : taux de champs manquants ─────────────
fields = ['genres', 'overview', 'tagline', 'budget', 'revenue', 'runtime', 'keywords', 'credits']
total  = es.count(index='movies_clean')['count']

rows = []
for field in fields:
    count = es.count(index='movies_clean', query={'exists': {'field': field}})['count']
    rows.append({'champ': field, 'renseigné': count, 'manquant': total - count,
                 'taux_renseigné': f"{count/total*100:.1f}%"})

df_quality = pd.DataFrame(rows)
print(df_quality.to_string(index=False))